# 验证获取的数据

本 Notebook 用于读取 `data_fetch_ccxtpro` 模块采集并保存在 Parquet 文件中的高频行情数据（Trade 和 Orderbook），并进行输出验证。

In [ ]:
import os
import pandas as pd
from glob import glob

# 设置数据根目录 (根据你的 fetch_config.yaml 确定)
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.getcwd()))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')

print(f"Data Directory: {DATA_DIR}")

## 1. 验证 Trade 逐笔交易数据

In [ ]:
trade_files = glob(os.path.join(DATA_DIR, 'trades', '**', '*.parquet'), recursive=True)

if not trade_files:
    print("未找到任何 Trade 数据文件！请确认采集程序正在运行并在生成数据。")
else:
    print(f"找到了 {len(trade_files)} 个 Trade 数据文件。读取最新文件：")
    latest_trade_file = sorted(trade_files, key=os.path.getmtime)[-1]
    print(f"读取文件: {latest_trade_file}")
    
    try:
        df_trade = pd.read_parquet(latest_trade_file)
        print(f"\n数据形状: {df_trade.shape}")
        display(df_trade.tail())
        
        print("\n--- 基础统计描述 ---")
        display(df_trade[['price', 'amount']].describe())
    except Exception as e:
        print(f"读取 Trade Parquet 文件失败: {e}")

## 2. 验证 Orderbook 订单簿深度数据

In [ ]:
ob_files = glob(os.path.join(DATA_DIR, 'orderbooks', '**', '*.parquet'), recursive=True)

if not ob_files:
    print("未找到任何 Orderbook 数据文件！请确认采集程序正在运行并在生成数据。")
else:
    print(f"找到了 {len(ob_files)} 个 Orderbook 数据文件。读取最新文件：")
    latest_ob_file = sorted(ob_files, key=os.path.getmtime)[-1]
    print(f"读取文件: {latest_ob_file}")
    
    try:
        df_ob = pd.read_parquet(latest_ob_file)
        print(f"\n数据形状: {df_ob.shape}")
        display(df_ob.tail())
        
        print("\n--- 抽取部分核心字段 (L1档位) ---")
        cols_to_show = ['local_ts', 'exchange_ts', 'bid_p_1', 'bid_q_1', 'ask_p_1', 'ask_q_1']
        display(df_ob[cols_to_show].tail())
    except Exception as e:
        print(f"读取 Orderbook Parquet 文件失败: {e}")

## 3. 手动体检：扫描并清理损坏的 Parquet 文件

由于早前版本在使用 `fastparquet` 引擎配合 `append=True` 原地追加时，破坏了原文件的尾部元数据（Footer metadata），这会导致后续读取时报 `Parquet magic bytes not found` 或 `Invalid column metadata` 错误。

运行下方的格子可以遍历扫描所有 `.parquet` 文件，手动找出那些读取失败的数据（可选将其**直接删除**，以保证新进来的数据和分析流程干净）。

In [ ]:
def check_and_clean_corrupted_files(data_dir, delete_corrupted=False):
    print(f"🔍 开始扫描目录: {data_dir}")
    all_parquet_files = glob(os.path.join(data_dir, '**', '*.parquet'), recursive=True)
    
    corrupted_files = []
    for file_path in all_parquet_files:
        try:
            # 首选使用 pyarrow 引擎测试读取，如果不报错说明健康
            pd.read_parquet(file_path, engine='pyarrow')
        except Exception as e:
            corrupted_files.append((file_path, str(e)))
            
    print(f"\n扫描完成！共扫描 {len(all_parquet_files)} 个文件。")
    
    if corrupted_files:
        print(f"⚠️ 发现 {len(corrupted_files)} 个损坏的 Parquet 文件：")
        for path, err in corrupted_files:
            print(f" - {path}\n   └─ 报错信息: {err[:150]}...")
            
            if delete_corrupted:
                os.remove(path)
                print("     ✅ 已删除该损坏文件")
    else:
        print("✅ 未发现任何损坏文件，所有 Parquet 均可正常读取！")

# 默认只检查不删除，如果确认想一次性清理历史损坏文件，将下方的 False 改为 True
check_and_clean_corrupted_files(DATA_DIR, delete_corrupted=False)

## 4. 读取指定 Parquet 文件进行分析

这里演示如何直接读取具体的 Parquet 文件进行定制化的验证或分析。

In [ ]:
import pandas as pd
import os

# 你可以按需修改交易所、交易对、日期及具体的文件名 (这里以 Binance BTC/USDT spot 为例)
specific_ob_file = os.path.join(DATA_DIR, 'orderbooks', 'market_type=spot', 'exchange=binance', 'symbol=BTC_USDT', 'date=2026-02-24', '16.parquet')

if os.path.exists(specific_ob_file):
    print(f"正在读取指定的 Orderbook 文件: {specific_ob_file}")
    df_specific_ob = pd.read_parquet(specific_ob_file)
    print(f"Orderbook 数据形状: {df_specific_ob.shape}")
    display(df_specific_ob.head())
    
    # 计算并输出 Orderbook 数据的时间跨度
    if 'local_ts' in df_specific_ob.columns:
        min_ts = pd.to_datetime(df_specific_ob['local_ts'].min(), unit='ms')
        max_ts = pd.to_datetime(df_specific_ob['local_ts'].max(), unit='ms')
        duration = max_ts - min_ts
        print(f"\n⏰ Orderbook 数据时间跨度:")
        print(f"  开始时间: {min_ts}")
        print(f"  结束时间: {max_ts}")
        print(f"  总时段: {duration}")
else:
    print(f"找不到 Orderbook 文件: {specific_ob_file}")

# 验证 Trade 文件
specific_trade_file = os.path.join(DATA_DIR, 'trades', 'market_type=spot', 'exchange=binance', 'symbol=BTC_USDT', 'date=2026-02-24', '16.parquet')
if os.path.exists(specific_trade_file):
    print(f"\n{'-'*40}\n正在读取指定的 Trade 文件: {specific_trade_file}")
    df_specific_trade = pd.read_parquet(specific_trade_file)
    print(f"Trade 数据形状: {df_specific_trade.shape}")
    display(df_specific_trade.head())
    
    # 计算并输出 Trade 数据的时间跨度
    if 'local_ts' in df_specific_trade.columns:
        min_ts = pd.to_datetime(df_specific_trade['local_ts'].min(), unit='ms')
        max_ts = pd.to_datetime(df_specific_trade['local_ts'].max(), unit='ms')
        duration = max_ts - min_ts
        print(f"\n⏰ Trade 数据时间跨度:")
        print(f"  开始时间: {min_ts}")
        print(f"  结束时间: {max_ts}")
        print(f"  总时段: {duration}")
else:
    print(f"\n找不到 Trade 文件: {specific_trade_file}")
